# 01 — Data Acquisition

This notebook retrieves cancer cohort metadata from the NCI Genomic Data Commons (GDC) API for downstream survival analysis.

We use the GDC API because it supports programmatic access to cases and related clinical fields, and conveniently has an API to work with. We will collect a raw case-level table that can later be cleaned into analysis variables such as survival time, censoring indicator, age, sex, and disease type.

In [1]:
!pip install -r ../code/requirements.txt

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import requests
import pandas as pd
import json
from pathlib import Path

# Create output folders
Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

## Configuration

We begin with a single TCGA set (BRCA for breast cancer) so the workflow is easy to debug. We can change this later

In [3]:
BASE_URL = "https://api.gdc.cancer.gov/cases"
PROJECT_ID = "TCGA-BRCA"
SIZE = 5000  # large enough for most TCGA cohorts

## Define the fields to retrieve

We request identifiers, project metadata, diagnosis information, demographic variables, and follow-up / vital status fields that are useful for survival analysis. We wanna pull raw first, and then construct covariates later

In [4]:
fields = [
    "id",
    "case_id",
    "submitter_id",
    "project.project_id",

    "demographic.race",
    "demographic.gender",
    "demographic.ethnicity",
    "demographic.vital_status", # used to be a diagnosis subobject, see recent documentation
    "demographic.days_to_death",# used to be a diagnosis subobject, see recent documentation

    "diagnoses.days_to_last_follow_up",
    "diagnoses.age_at_diagnosis",
    "diagnoses.primary_diagnosis",
    "diagnoses.ajcc_pathologic_stage",
    "diagnoses.tumor_grade"
]

In [5]:
filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "project.project_id",
                "value": [PROJECT_ID]
            }
        }
    ]
}

params = {
    "filters": json.dumps(filters),
    "fields": ",".join(fields),
    "format": "JSON",
    "size": SIZE,
    "expand": "diagnoses"
}

In [6]:
response = requests.get(BASE_URL, params=params, timeout=60)
response.raise_for_status()

payload = response.json()
hits = payload["data"]["hits"]

print(f"Retrieved {len(hits)} case records from {PROJECT_ID}")

Retrieved 1098 case records from TCGA-BRCA


## Flatten the JSON payload

In [7]:
df = pd.json_normalize(hits)
df.head()

,id,case_id,submitter_id,diagnoses,project.project_id,demographic.race,demographic.gender,demographic.ethnicity,demographic.vital_status,demographic.days_to_death
0,e3935ce4-64d3-4a66-ba11-d308b844b410,e3935ce4-64d3-4a66-ba11-d308b844b410,TCGA-E9-A5FL,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not hispanic or latino,Alive,NaN
1,e3b555aa-7f0a-49c6-9b13-182c61a144c1,e3b555aa-7f0a-49c6-9b13-182c61a144c1,TCGA-A2-A1G4,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not hispanic or latino,Alive,NaN
2,17ca61a2-607a-45ff-88fa-ef72e80bf891,17ca61a2-607a-45ff-88fa-ef72e80bf891,TCGA-BH-A0HF,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not reported,Alive,NaN
3,7d681cc6-689d-41c8-9e84-e13733089ec9,7d681cc6-689d-41c8-9e84-e13733089ec9,TCGA-AR-A1AS,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,asian,female,not reported,Alive,NaN
4,17f275c1-a0d4-487d-8f02-ea279584b4cd,17f275c1-a0d4-487d-8f02-ea279584b4cd,TCGA-D8-A13Y,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not hispanic or latino,Alive,NaN


In [8]:
print(df.columns.tolist())
print(df.shape)

['id', 'case_id', 'submitter_id', 'diagnoses', 'project.project_id', 'demographic.race', 'demographic.gender', 'demographic.ethnicity', 'demographic.vital_status', 'demographic.days_to_death']
(1098, 10)


## Save raw cohort table

In [9]:
raw_path = Path("../data/raw") / f"{PROJECT_ID.lower()}_cases_raw.csv"
df.to_csv(raw_path, index=False)
print(f"Saved raw data to: {raw_path}")

Saved raw data to: ../data/raw/tcga-brca_cases_raw.csv


The GDC API returned 1098 case records for the selected TCGA cohort. Core demographic metadata were available as top-level fields, while diagnosis and survival-related information were nested inside the diagnoses field. This indicates that additional flattening is required before constructing analysis variables such as survival time, censoring indicator, age at diagnosis, and tumor stage.